In [1]:
from pathlib import Path
import numpy as np, pandas as pd, joblib, json
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, average_precision_score, classification_report, confusion_matrix


In [2]:
DATA, MODELS = Path("../data/processed/spam"), Path("../models")
MODELS.mkdir(exist_ok=True)

train, val = pd.read_csv(DATA/"train.csv"), pd.read_csv(DATA/"val.csv")
X_train, y_train = train["clean_text"].fillna(""), train["spam"].values
X_val,   y_val   = val["clean_text"].fillna(""),   val["spam"].values
print(train.shape, val.shape, "| spam in train:", round(y_train.mean()*100, 1), "%")

(3600, 4) (771, 4) | spam in train: 12.2 %


In [3]:
features = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True)),
])

model = Pipeline([
    ("tfidf", features),
    ("clf", LogisticRegression(C=10, solver="liblinear", class_weight="balanced")),
])
model.fit(X_train, y_train)

,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformer_list,"[('word', ...), ('char', ...)]"
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,input,'content'
,encoding,'utf-8'


In [4]:
val_scores = model.predict_proba(X_val)[:, 1]

best_t, best_f1 = 0.5, 0
for t in np.arange(0.05, 0.96, 0.01):
    f1 = f1_score(y_val, val_scores >= t, zero_division=0)
    if f1 > best_f1:
        best_t, best_f1 = round(float(t), 2), f1

val_pred = (val_scores >= best_t).astype(int)
print("threshold:", best_t, "| val spam F1:", round(best_f1, 4),
      "| PR-AUC:", round(average_precision_score(y_val, val_scores), 4))
print(classification_report(y_val, val_pred, target_names=["ham", "spam"]))
print("confusion matrix [rows = truth, cols = predicted]:")
print(confusion_matrix(y_val, val_pred))

threshold: 0.54 | val spam F1: 0.9838 | PR-AUC: 0.9959
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00       677
        spam       1.00      0.97      0.98        94

    accuracy                           1.00       771
   macro avg       1.00      0.98      0.99       771
weighted avg       1.00      1.00      1.00       771

confusion matrix [rows = truth, cols = predicted]:
[[677   0]
 [  3  91]]


In [5]:
import re
def clean_spam(t):                                   # same as notebook 01
    t = str(t).lower()
    t = re.sub(r"http\S+|www\.\S+", " urltoken ", t)
    t = re.sub(r"\d{5,}", " longnumber ", t)
    return re.sub(r"\s+", " ", t).strip()

samples = [
    "hey are we still meeting for lunch tomorrow?",
    "WINNER!! You have won a free prize, claim now at www.win-cash.com",
    "huge sale 50% off, order now and call 9876543210",
    "bhai notes bhej dena, kal exam hai",
]
s = model.predict_proba(pd.Series(samples).apply(clean_spam))[:, 1]
print(pd.DataFrame({"spam_score": s.round(2)}, index=samples).to_string())

                                                                   spam_score
hey are we still meeting for lunch tomorrow?                             0.00
WINNER!! You have won a free prize, claim now at www.win-cash.com        1.00
huge sale 50% off, order now and call 9876543210                         1.00
bhai notes bhej dena, kal exam hai                                       0.01


In [6]:
joblib.dump({"model": model, "threshold": best_t}, MODELS / "spam_model.joblib")
json.dump({"threshold": best_t}, open(MODELS / "spam_threshold.json", "w"))